# IDH v2 · Tumour segmentation on Kaggle (GPU)

Trains the cross-fitted SegResNet of [DinoV2_IDH](https://github.com/Fuzzonaut/DinoV2_IDH)
and writes a predicted tumour mask for every UCSF-PDGM patient. About 3–4 hours on a T4.

**Settings (right panel):** Accelerator = GPU (T4 or P100), Internet = On.
Then **Save Version → Save & Run All (Commit)**. When it has finished, download
`idh_v2_segmentation.zip` from the *Output* tab.

In [ ]:
!git clone -q -b v2-segmentation https://github.com/Fuzzonaut/DinoV2_IDH.git /tmp/DinoV2_IDH
%cd /tmp/DinoV2_IDH
!pip install -q "monai>=1.5"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1 · Data (~6 GB: FLAIR, T1, T1c, T2 and the expert tumour masks)

In [ ]:
!python prepare_data.py --data-dir /tmp/data

## 2 · Cross-fitted segmentation

3 folds over the 420 development patients (each fold's model segments the patients it was not
trained on); the 75 test patients are segmented by the 3-model ensemble. Finished folds are kept in
`/kaggle/working/segmentation_model`, so a restarted run continues where it stopped.

In [ ]:
!python train_segmentation.py --cohort /tmp/data/cohort.csv --cache-dir /tmp/seg_cache \
    --masks-dir /kaggle/working/pred_masks --model-dir /kaggle/working/segmentation_model \
    --results-dir /kaggle/working/results_v2 --workers 4

## 3 · Package the output

In [ ]:
%cd /kaggle/working
!zip -qr idh_v2_segmentation.zip pred_masks segmentation_model results_v2
!rm -rf pred_masks
!ls -la /kaggle/working